In [1]:
import json

catalog = {
    "gloves": {"stock": 7, "unit_cents": 325},
    "goggles": {"stock": 4, "unit_cents": 850},
}
for item, fields in catalog.items():
    print(item, fields)
print("Items:", len(catalog))

gloves {'stock': 7, 'unit_cents': 325}
goggles {'stock': 4, 'unit_cents': 850}
Items: 2


A lookup checks the actual data. Guessing could be wrong.

In [2]:
def stock_lookup(item):
    """Return available units for a catalog item."""
    return {"item": item, "available": catalog[item]["stock"]}

def supply_quote(item, units):
    """Calculate cents for 1 through 20 units; reserve nothing."""
    total = catalog[item]["unit_cents"] * units
    return {"item": item, "units": units, "total_cents": total}

print(stock_lookup("gloves"))
print(supply_quote("gloves", 6))

{'item': 'gloves', 'available': 7}
{'item': 'gloves', 'units': 6, 'total_cents': 1950}


stock_lookup checks how many we have. supply_quote checks the price.

In [3]:
item_rule = {"type": "string", "enum": list(catalog)}
stock_schema = {
    "type": "object",
    "properties": {"item": item_rule},
    "required": ["item"], "additionalProperties": False,
}
quote_schema = {
    "type": "object",
    "properties": {
        "item": item_rule,
        "units": {"type": "integer", "minimum": 1, "maximum": 20},
    },
    "required": ["item", "units"], "additionalProperties": False,
}
print(json.dumps(quote_schema, indent=2))

{
  "type": "object",
  "properties": {
    "item": {
      "type": "string",
      "enum": [
        "gloves",
        "goggles"
      ]
    },
    "units": {
      "type": "integer",
      "minimum": 1,
      "maximum": 20
    }
  },
  "required": [
    "item",
    "units"
  ],
  "additionalProperties": false
}


No. The schema just sets the rule. The program has to check it.

In [4]:
registry = {
    "stock_lookup": {"run": stock_lookup,
        "description": "Read available workshop units for one item.",
        "schema": stock_schema},
    "supply_quote": {"run": supply_quote,
        "description": "Calculate an estimated cost in cents only; does not check availability, reserve inventory, submit an order, or purchase anything.",
        "schema": quote_schema},
}
definitions = [
    {"name": name, "description": spec["description"], "input_schema": spec["schema"]}
    for name, spec in registry.items()
]
prompt = ('For a tool request return JSON with name and arguments. '
          'For a direct answer return JSON with answer. Available tools: '
          + json.dumps(definitions))
print([tool["name"] for tool in definitions])

['stock_lookup', 'supply_quote']


supply_quote gives a price only. It does not order or reserve anything.

In [5]:
def parse_call(text):
    call = json.loads(text)
    if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
        raise ValueError("Expected name and arguments only")
    name, args = call["name"], call["arguments"]
    if not isinstance(name, str) or name not in registry:
        raise ValueError("Unknown tool")
    schema = registry[name]["schema"]
    if not isinstance(args, dict) or set(args) != set(schema["required"]):
        raise ValueError("Missing or extra argument")
    for key, rule in schema["properties"].items():
        value = args[key]
        if rule["type"] == "string":
            if not isinstance(value, str) or value not in rule["enum"]:
                raise ValueError("Invalid item")
        elif rule["type"] == "integer":
            if type(value) is not int:
                raise ValueError("Quantity must be an integer")
            if not rule["minimum"] <= value <= rule["maximum"]:
                raise ValueError("Quantity outside allowed range")
    return call

proposal = '{"name":"supply_quote","arguments":{"item":"gloves","units":6}}'
print(parse_call(proposal))

{'name': 'supply_quote', 'arguments': {'item': 'gloves', 'units': 6}}


Parsing checks if the JSON works. Validation checks if the info is allowed.

In [6]:
drafts = []
def draft_request(item, units):
    draft = {"item": item, "units": units}
    drafts.append(draft)
    return {"draft_number": len(drafts), "status": "draft only"}

registry["draft_request"] = {
    "run": draft_request, "description": "Create a local supply draft.",
    "schema": quote_schema,
}
approval_tools = {"draft_request"}

def execute_call(text, approved=False):
    try:
        call = parse_call(text)
        if call["name"] in approval_tools and not approved:
            return {"ok": False, "error": "Approval required"}
        result = registry[call["name"]]["run"](**call["arguments"])
        return {"ok": True, "result": result}
    except (ValueError, TypeError, KeyError) as error:
        return {"ok": False, "error": str(error)}

print(execute_call(proposal))

{'ok': True, 'result': {'item': 'gloves', 'units': 6, 'total_cents': 1950}}


The app needs real approval. The model cannot approve itself.

In [7]:
fixtures = [
    proposal,
    "not JSON",
    '{"name":"erase_all","arguments":{}}',
    '{"name":"supply_quote","arguments":{"item":"gloves","units":true}}',
    '{"name":"supply_quote","arguments":{"item":"gloves","units":21}}',
    '{"name":"draft_request","arguments":{"item":"gloves","units":6}}',
]
for text in fixtures:
    print(execute_call(text))
print("Drafts before approval:", len(drafts))
print(execute_call(fixtures[-1], approved=True))
print("Drafts after approval:", len(drafts))

{'ok': True, 'result': {'item': 'gloves', 'units': 6, 'total_cents': 1950}}
{'ok': False, 'error': 'Expecting value: line 1 column 1 (char 0)'}
{'ok': False, 'error': 'Unknown tool'}
{'ok': False, 'error': 'Quantity must be an integer'}
{'ok': False, 'error': 'Quantity outside allowed range'}
{'ok': False, 'error': 'Approval required'}
Drafts before approval: 0
{'ok': True, 'result': {'draft_number': 1, 'status': 'draft only'}}
Drafts after approval: 1


erase_all is not a real tool, and 21 units is over the limit.

In [8]:
history = [{"role": "user", "content": "Quote six gloves."}]
history.append({"role": "assistant", "call_id": "quote-1", "content": proposal})
observation = execute_call(proposal)
history.append({"role": "tool", "call_id": "quote-1", "content": json.dumps(observation)})
if observation["ok"]:
    cents = observation["result"]["total_cents"]
    answer = f"Six gloves cost ${cents / 100:.2f}. Stock is unverified."
else:
    answer = "The quote failed; no price is confirmed."
history.append({"role": "assistant", "content": answer})
print(answer)
print([message["role"] for message in history])

Six gloves cost $19.50. Stock is unverified.
['user', 'assistant', 'tool', 'assistant']


Use stock_lookup to see if six gloves are available.

In [9]:
native_fixture = {
    "id": "stock-2", "type": "function",
    "function": {"name": "stock_lookup", "arguments": '{"item":"gloves"}'},
}
def normalize_native(record):
    function = record["function"]
    args = function["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    return json.dumps({"name": function["name"], "arguments": args})

native_result = execute_call(normalize_native(native_fixture))
tool_reply = {"role": "tool", "tool_call_id": native_fixture["id"],
              "content": json.dumps(native_result)}
print(tool_reply)

{'role': 'tool', 'tool_call_id': 'stock-2', 'content': '{"ok": true, "result": {"item": "gloves", "available": 7}}'}


Both need the same checks so bad requests cannot get through.

In [10]:
def discover(query):
    words = set(query.lower().split())
    tags = {
        "stock_lookup": {"stock", "available", "inventory"},
        "supply_quote": {"cost", "price", "quote"},
        "draft_request": {"draft", "request"},
    }
    return [name for name, hints in tags.items() if words & hints]

print(discover("available gloves"))
print(discover("cost of gloves"))
print(discover("hello"))

['stock_lookup']
['supply_quote']
[]


I would search again or show more tools instead of guessing.

In [11]:
def simulated_server(method, params):
    if method == "tools/list":
        return {"tools": [
            {"name": name, "description": spec["description"],
             "inputSchema": spec["schema"]}
            for name, spec in registry.items()
        ]}
    if method == "tools/call":
        return execute_call(json.dumps(params))
    return {"error": "Unsupported method in simulation"}

listed = simulated_server("tools/list", {})
print("Discovered:", [tool["name"] for tool in listed["tools"]])
params = {"name": "stock_lookup", "arguments": {"item": "goggles"}}
print(simulated_server("tools/call", params))
print("OFFLINE ROLE PLAY ONLY")

Discovered: ['stock_lookup', 'supply_quote', 'draft_request']
{'ok': True, 'result': {'item': 'goggles', 'available': 4}}
OFFLINE ROLE PLAY ONLY


Without MCP it is 12 connections. With shared adapters it is 7. Real setups still need security and maintenance.

In [12]:
skill_demo = {
    "name": "workshop-supply-check",
    "description": "Use when checking a workshop supply request.",
    "instructions": (
        "Look up stock. Calculate the quote. Report both results. "
        "Ask the user before creating any draft."
    ),
    "resources": {"report-format.md": "Item, availability, cost, decision"},
}
print("Layer 1:", skill_demo["name"], skill_demo["description"])
print("Layer 2:", skill_demo["instructions"])
print("Layer 3:", skill_demo["resources"]["report-format.md"])
print("Instructions alone do not enforce authorization.")

Layer 1: workshop-supply-check Use when checking a workshop supply request.
Layer 2: Look up stock. Calculate the quote. Report both results. Ask the user before creating any draft.
Layer 3: Item, availability, cost, decision
Instructions alone do not enforce authorization.


I would add more details to report-format.md and check stock inside execute_call.

In [13]:
goggles_stock = stock_lookup("goggles")
goggles_quote = supply_quote("goggles", 5)
shortage = max(0, 5 - goggles_stock["available"])
print("Goggles available:", goggles_stock["available"])
print("Requested:", 5)
print("Shortage:", shortage)
print("Quote:", goggles_quote["total_cents"], "cents")
print(f"Quoted cost: ${goggles_quote['total_cents'] / 100:.2f}")
print("Drafts unchanged:", len(drafts))

Goggles available: 4
Requested: 5
Shortage: 1
Quote: 4250 cents
Quoted cost: $42.50
Drafts unchanged: 1


We have 4 goggles but need 5, so we are short 1. Five cost $42.50. A quote is just a price, not an order.

In [14]:
boundary_cases = [
    ("Missing units", {"item": "gloves"}, False),
    ("Unexpected price", {"item": "gloves", "units": 6, "price": 1}, False),
    ("Invalid item tape", {"item": "tape", "units": 6}, False),
    ("Units zero", {"item": "gloves", "units": 0}, False),
    ("Units twenty", {"item": "gloves", "units": 20}, True),
    ("Units twenty-one", {"item": "gloves", "units": 21}, False),
    ("Units string six", {"item": "gloves", "units": "6"}, False),
    ("Units boolean true", {"item": "gloves", "units": True}, False),
]
drafts_before = len(drafts)
for label, arguments, predicted_success in boundary_cases:
    request = json.dumps({"name": "supply_quote", "arguments": arguments})
    result = execute_call(request)
    print(f"{label}: predicted={'success' if predicted_success else 'failure'}, actual={result}")
    assert result["ok"] == predicted_success
assert len(drafts) == drafts_before
print("All predictions matched. No drafts added.")

Missing units: predicted=failure, actual={'ok': False, 'error': 'Missing or extra argument'}
Unexpected price: predicted=failure, actual={'ok': False, 'error': 'Missing or extra argument'}
Invalid item tape: predicted=failure, actual={'ok': False, 'error': 'Invalid item'}
Units zero: predicted=failure, actual={'ok': False, 'error': 'Quantity outside allowed range'}
Units twenty: predicted=success, actual={'ok': True, 'result': {'item': 'gloves', 'units': 20, 'total_cents': 6500}}
Units twenty-one: predicted=failure, actual={'ok': False, 'error': 'Quantity outside allowed range'}
Units string six: predicted=failure, actual={'ok': False, 'error': 'Quantity must be an integer'}
Units boolean true: predicted=failure, actual={'ok': False, 'error': 'Quantity must be an integer'}
All predictions matched. No drafts added.


Only 20 works. Wrong items, missing info, and bad numbers fail. None of the tests make a draft. Bad descriptions can make the model pick the wrong tool. A short list might leave out the one it needs. These are just examples, not real model tests.

In [15]:
complete_history = [{"role": "user", "content": "Are six gloves available, and what do they cost?"}]
for call_id, name, args in [
    ("stock-1", "stock_lookup", {"item": "gloves"}),
    ("quote-2", "supply_quote", {"item": "gloves", "units": 6}),
]:
    request = json.dumps({"name": name, "arguments": args})
    complete_history.append({"role": "assistant", "call_id": call_id, "content": request})
    result = execute_call(request)
    complete_history.append({"role": "tool", "call_id": call_id, "content": json.dumps(result)})
stock_result = json.loads(complete_history[2]["content"])["result"]
quote_result = json.loads(complete_history[4]["content"])["result"]
final_answer = (f"{stock_result['available']} gloves are available, so the workshop has enough for six. "
                f"The quote for six gloves is ${quote_result['total_cents']/100:.2f}. "
                "No stock has been reserved and no order was placed.")
complete_history.append({"role": "assistant", "content": final_answer})
print(final_answer)
print("Transcript roles:", [message["role"] for message in complete_history])
print("Call IDs:", [m["call_id"] for m in complete_history if "call_id" in m])

7 gloves are available, so the workshop has enough for six. The quote for six gloves is $19.50. No stock has been reserved and no order was placed.
Transcript roles: ['user', 'assistant', 'tool', 'assistant', 'tool', 'assistant']
Call IDs: ['stock-1', 'stock-1', 'quote-2', 'quote-2']


In [16]:
def bounded_scripted_loop(records, max_calls=2):
    results = []
    used = 0
    for record in records:
        if "answer" in record:
            return {"answer": record["answer"], "calls_used": used, "observations": results}
        if used >= max_calls:
            return {"error": "Call budget exhausted", "calls_used": used, "observations": results}
        results.append(execute_call(json.dumps(record)))
        used += 1
    return {"error": "No answer record", "calls_used": used, "observations": results}

demo_records = [
    {"name": "stock_lookup", "arguments": {"item": "gloves"}},
    {"name": "supply_quote", "arguments": {"item": "gloves", "units": 6}},
    {"answer": "Seven gloves are available; six cost $19.50. No reservation."},
]
print(bounded_scripted_loop(demo_records))
over_budget = demo_records[:2] + [
    {"name": "stock_lookup", "arguments": {"item": "goggles"}}
]
print(bounded_scripted_loop(over_budget))

{'answer': 'Seven gloves are available; six cost $19.50. No reservation.', 'calls_used': 2, 'observations': [{'ok': True, 'result': {'item': 'gloves', 'available': 7}}, {'ok': True, 'result': {'item': 'gloves', 'units': 6, 'total_cents': 1950}}]}
{'error': 'Call budget exhausted', 'calls_used': 2, 'observations': [{'ok': True, 'result': {'item': 'gloves', 'available': 7}}, {'ok': True, 'result': {'item': 'gloves', 'units': 6, 'total_cents': 1950}}]}


The loop stops when it gets an answer or hits the two-call limit.